In [8]:
# Ensure 'df' is loaded and has 'sentiment'
import pandas as pd

# Load merged data if needed
df = pd.read_csv('../data/merged_stock_news.csv')
df['sentiment'] = df['headline'].astype(str).apply(lambda x: TextBlob(x).sentiment.polarity)
df['date'] = pd.to_datetime(df['date'])

# Aggregate: daily average sentiment per stock
daily_sentiment = (
    df.groupby(['stock', 'date'])['sentiment']
    .mean()
    .reset_index()
    .rename(columns={'sentiment': 'daily_sentiment'})
)

print(daily_sentiment.head())


  stock       date  daily_sentiment
0  AAPL 2024-01-12         0.000000
1  AAPL 2024-01-31         0.000000
2  AAPL 2024-02-02         0.433333
3  AAPL 2024-02-12         0.000000
4  AAPL 2024-03-15         0.433333


In [9]:
import pandas as pd
import glob
import os

# Load all stock price data
stock_files = glob.glob(os.path.join('../data', 'yfinance_data', '*_historical_data*.csv'))
dfs_stock = []

for file in stock_files:
    df_stock = pd.read_csv(file)
    stock_name = os.path.basename(file).split('_')[0]  # Extract stock ticker
    df_stock['stock'] = stock_name
    df_stock['Date'] = pd.to_datetime(df_stock['Date'])
    dfs_stock.append(df_stock)

df_prices = pd.concat(dfs_stock, ignore_index=True)

# Merge with daily sentiment
merged_data = pd.merge(
    df_prices,
    daily_sentiment,
    left_on=['stock', 'Date'],
    right_on=['stock', 'date'],
    how='left'
).drop(columns=['date'])

print(merged_data.head())


        Date      Open      High       Low     Close  Adj Close     Volume  \
0 1980-12-12  0.128348  0.128906  0.128348  0.128348   0.098943  469033600   
1 1980-12-15  0.122210  0.122210  0.121652  0.121652   0.093781  175884800   
2 1980-12-16  0.113281  0.113281  0.112723  0.112723   0.086898  105728000   
3 1980-12-17  0.115513  0.116071  0.115513  0.115513   0.089049   86441600   
4 1980-12-18  0.118862  0.119420  0.118862  0.118862   0.091630   73449600   

   Dividends  Stock Splits stock  daily_sentiment  
0        0.0           0.0  AAPL              NaN  
1        0.0           0.0  AAPL              NaN  
2        0.0           0.0  AAPL              NaN  
3        0.0           0.0  AAPL              NaN  
4        0.0           0.0  AAPL              NaN  
